# 🌊 Flash Flood Detection in Arizona Using VideoDB RTStream + TwelveLabs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/integrations/twelvelabs/Flash_Flood_Detection_TwelveLabs.ipynb)

## 📖 Storytime: Why This Matters

The stunning Arizona deserts, known for their dry riverbeds and scenic beauty, hide a deadly risk.  
During the summer monsoon, sudden torrential rains can trigger **flash floods** in these seemingly harmless dry zones — with little or no warning.

Conventional alert systems relying on rain gauges or weather satellites often fail to deliver timely, location-specific warnings. By the time a danger alert is sent, it might already be too late.

**But we have a smarter way.**

With **VideoDB RTStream**, we can install real-time cameras near flood-prone areas and let AI continuously monitor the visuals powered by **TwelveLabs Pegasus 1.2**.  
As soon as the AI detects signs of a flash flood — like a sudden surge of water through dry land — it can instantly send alerts, giving local authorities and tourists precious moments to act.

---

## 🚀 What You’ll Build in This Notebook

In this notebook, we’ll create a real-time flash flood detection system using **VideoDB RTStream**.  
By the end of this demo, you’ll learn how to:
- Connect a live video stream to VideoDB
- Use AI to continuously analyze scenes for signs of a flash flood
- Detect a **flash flood event**
- Trigger a real-time alert when detected

Let’s build it together!



---

## 📦 Step 1: Install Dependencies

Before setting up our AI-powered flood monitor, let’s install the required VideoDB SDK.

In [ ]:
!pip install -q videodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.9/95.9 kB 4.1 MB/s eta 0:00:00


---
## 📦 Step 2: Connect to VideoDB

Let's connect to VideoDB's API using your credentials to prepare for stream monitoring.

Please enter your `VIDEO_DB_API_KEY` in the input box that appears below after you run this cell.

Your input will be masked.


In [ ]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")

os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("Connected to VideoDB securely!")

Please enter your VideoDB API Key: ··········
Connected to VideoDB securely!


---

## 📦 Step 3: Connect to the Arizona Flood RTSP Stream

Connect to the live camera stream monitoring a flood-prone desert area.

In this demo, the stream is running at `rtsp://samples.rts.videodb.io:8554/floods`.

In [ ]:
rtsp_url = "rtsp://samples.rts.videodb.io:8554/floods"
flood_stream = coll.connect_rtstream(
    name="Arizona Flood Stream",
    url=rtsp_url,
    store=True,
)
print(flood_stream)


RTStream(id=rts-019fa33a-b352-7c60-a51e-007acc0f652d, name=Arizona Flood Stream, collection_id=c-81fc6459-fe30-44ac-8c5b-ea0898c2e152, created_at=2026-07-27T10:59:21.042258+00:00, sample_rate=30, status=connected, stream_url=None, player_url=None)


#### Let us list all the rtstreams in our collection.

In [ ]:
def list_rtstreams():
    for rtstream in coll.list_rtstreams():
        print(f"""RTStream:
        ID            : {rtstream.id}
        Name          : {rtstream.name}
        Collection ID : {rtstream.collection_id}
        Created At    : {rtstream.created_at}
        Sample Rate   : {rtstream.sample_rate}
        Status        : {rtstream.status}
        """)
        print("-" * 80)

list_rtstreams()

RTStream:
        ID            : rts-019fa33a-b352-7c60-a51e-007acc0f652d
        Name          : Arizona Flood Stream
        Collection ID : c-81fc6459-fe30-44ac-8c5b-ea0898c2e152
        Created At    : 2026-07-27T10:59:21.042258+00:00
        Sample Rate   : 30
        Status        : connected
        
--------------------------------------------------------------------------------
RTStream:
        ID            : rts-019fa32f-be03-78d3-8acf-97ccff3f2940
        Name          : RoadCam Violation Stream
        Collection ID : c-81fc6459-fe30-44ac-8c5b-ea0898c2e152
        Created At    : 2026-07-27T10:47:22.883275+00:00
        Sample Rate   : 30
        Status        : stopped
        
--------------------------------------------------------------------------------
RTStream:
        ID            : rts-019fa26c-02b3-7ae3-a62f-0c9b2139ef69
        Name          : RoadCam Violation Stream
        Collection ID : c-81fc6459-fe30-44ac-8c5b-ea0898c2e152
        Created At    : 2026-


#### If you have already connected the stream, run the below cell with the **rtstream id** to reconnect.

In [ ]:
# flood_stream = coll.get_rtstream("")

In [ ]:
# flood_stream.stop()
# print("Stream stopped")


In [ ]:
# To start the stream
# flood_stream.start()

---
### 👀 Let's have a look at the riverbed

#### 📺 Helper Functions: Search and Display

This cell contains all the utility functions to search, fetch, and visualize video streams. You don't need to modify this code.

In [ ]:
# To display the stream with relevant information

from IPython.display import HTML
import re
import time
from datetime import datetime, UTC
from videodb import play_stream


def display_stream(video_url, video_name="🎥 Camera Feed"):
    match = re.search(r"/(\d{16})-(\d{16})\.m3u8", video_url)
    if match:
        start_ts = int(match.group(1)) / 1e6
        end_ts = int(match.group(2)) / 1e6
        start_time = datetime.fromtimestamp(start_ts, UTC).strftime("%Y-%m-%d %H:%M:%S")
        end_time = datetime.fromtimestamp(end_ts, UTC).strftime("%Y-%m-%d %H:%M:%S")
        time_range = f"{start_time} → {end_time} UTC"
    else:
        time_range = "Time Unknown"

    video_player_html = play_stream(video_url)

    return HTML(f"""
    <div style="position:relative;width:640px;">
      {video_player_html._repr_html_() if hasattr(video_player_html, "_repr_html_") else video_player_html}
      <div style="position:absolute;top:10px;left:10px;background:rgba(0,0,0,0.6);color:#fff;padding:6px 12px;border-radius:4px;font-size:13px;font-family:sans-serif;">
        <strong>{video_name}</strong><br>{time_range}
      </div>
    </div>
    """)


# To dynamically set the display duration


def prompt_to_time(prompt):
    now = int(time.time())
    prompt = (
        f"It's {now} in epoch seconds. "
        f"Convert the phrase '{prompt}' into JSON "
        f'with keys "from" and "to" (both epoch seconds)'
    )

    result = coll.generate_text(
        prompt=prompt,
        model_name="pro",
        response_type="json",
    )
    output = result.get("output", {})
    return output.get("from"), output.get("to")


# To fetch stream


def fetch_stream(rtstream):
    _from, to = prompt_to_time("Show me last 5 mins")
    rtstream.generate_stream(_from, to)
    return rtstream.stream_url



#### 🔗 Get & Display Recent Stream

This cell uses the helper functions above to fetch and display the last few minutes of the stream.

In [ ]:
# To get last few minutes stream link

video_url = fetch_stream(flood_stream)

video_name = "🌊 Arizona Desert · Flash Flood Detection"
display_stream(video_url , video_name)


---

## 📦 Step 4: Understand Scenes and Detect Flash Floods

We’ll continuously analyze five-second windows, preserve a named flood-scene output, and index that output for records and alerts.

The AI will look for sudden visual cues of water flooding dry land and describe them.


In [ ]:
flood_understanding = flood_stream.understand(
    segmentation={"type": "time", "window": "5s"},
    analyzers=[
        {
            "type": "vlm",
            "name": "flood_scene",
            "sampling": {"frame_count": 3},
            "config": {
                "prompt": "Monitor the dry riverbed and surrounding area. If moving water is detected across the land, identify it as a flash flood and describe the scene.",
                "model": "twelvelabs-pegasus-1.2",

            },
        }
    ],
    store=True,
)
flood_scene_output = flood_understanding.outputs.get("flood_scene")
flood_understanding_id = flood_understanding.id
print("Continuous understanding ID:", flood_understanding_id)


Continuous understanding ID: und-2374a340228731ff


### Create the Flood Detection Index

Create a continuous semantic index from the flood-scene output for records and alerts.


In [ ]:
flood_index = flood_stream.index(
    source=flood_scene_output,
    name="Flash_Flood_Detection_Index",
    use_for=["semantic"],
)
flood_index_id = flood_index.id
print("Continuous index ID:", flood_index_id)


Continuous index ID: idx-51e7df660f56f036


#### Let us list the continuous indexes created on our rtstream.

In [ ]:
def list_rtstream_indexes(rtstream):
    # List continuous stream indexes
    rtstream_indexes = rtstream.list_indexes()
    for rtstream_index in rtstream_indexes:

        print(f"""RTStreamIndex:
            Index ID       : {rtstream_index.id}
            RTStream ID    : {rtstream_index.rtstream_id}
            Name           : {rtstream_index.name}
            Status         : {rtstream_index.status}
            Use For        : {rtstream_index.use_for}
            Understanding : {rtstream_index.source_understanding_id}
            Output        : {rtstream_index.output}
        """)
        print("-" * 80)

list_rtstream_indexes(flood_stream)

RTStreamIndex:
            Index ID       : idx-51e7df660f56f036
            RTStream ID    : rts-019fa33a-b352-7c60-a51e-007acc0f652d
            Name           : Flash_Flood_Detection_Index
            Status         : running
            Use For        : ['semantic']
            Understanding : und-2374a340228731ff
            Output        : flood_scene
        
--------------------------------------------------------------------------------



#### If you have already created the continuous jobs, run the below cell with your **understanding id** and **index id** to reconnect.

In [ ]:
# flood_understanding_id = ""
# flood_index_id = ""
# flood_understanding = flood_stream.get_understanding(flood_understanding_id)
# flood_index = flood_stream.get_index(flood_index_id)


In [ ]:
# To stop the index
# flood_index.stop()

In [ ]:
# To start the index
# flood_index.start()

---
### Let us inspect records from the continuous flood index

In [ ]:
import json
import time

range_end = time.time()
range_start = range_end - (15 * 60)
crib_records_payload = flood_index.get_records(
    start=range_start,
    end=range_end,
    page=1,
    page_size=5,
)
print(json.dumps(crib_records_payload, indent=2, default=str))


{
  "next_page": true,
  "records": [
    {
      "description": "Flash flood detected.\n\nMoving water is rushing through the dry riverbed, spilling over smooth rock shelves and funneling through narrow channels between the red sandstone. The flow looks fast and shallow in places, with white froth forming where it drops over ledges. Sparse desert shrubs line the higher banks under a cloudy sky, while the rocky channel below is actively being swept by water.",
      "end": 1785150173.114971,
      "start": 1785150168.094955
    },
    {
      "description": "Flash flood detected.\n\nMoving water is rushing through the dry riverbed, flowing rapidly between the rocks and across the channel. The scene shows shallow but energetic water churning over the reddish rock surface, with small cascades and white froth forming in the narrow cuts. The surrounding area is a rocky, arid landscape with sparse shrubs and no visible standing water beyond the channel, indicating a sudden runoff event.",
 

---
### 🔌 Step 4.5: Connect WebSocket for Real-Time Alerts

Before creating alerts, connect to VideoDB's WebSocket server.


In [ ]:
import asyncio

ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()

print(f"WebSocket connected!")
print(f"Connection ID: {ws.connection_id}")


INFO:videodb.websocket_client:WebSocket connected with ID: gVXiLA4Q8eO4KEiMGA==


WebSocket connected!
Connection ID: gVXiLA4Q8eO4KEiMGA==


## 🎤 Audio Indexing & Transcription

VideoDB RTStream also supports audio analysis for streams with audio content. You can:
- **Index Audio**: Extract structured information from audio streams using AI prompts
- **Start Transcription**: Get real-time speech-to-text transcription

These features work alongside visual indexing to provide comprehensive stream analysis.

In [ ]:
# # Audio Indexing Example (if stream has audio)
# audio_index = flood_stream.index_audio(
#     prompt="Detect sounds of rushing water, wind, rain, and warning sirens. Extract information about weather conditions and emergency alerts.",
#     batch_config={"type": "time", "value": 30},  # Segment every 30 seconds
#     name="Flood_Audio_Index"
# )

### Start Real-Time Transcription (Optional)

Start transcription and inspect the resulting speech-to-text records.

In [ ]:
# # Real-time Transcription Example
# # Get speech-to-text transcription in real-time
# flood_stream.start_transcript(ws_connection_id=ws.connection_id)
#
# # Poll transcript data
# transcript = flood_stream.get_transcript(
#     start=0,
#     end=None,
#     page=1,
#     page_size=100
# )
# print("Transcript data:", transcript)
#
# # Stop transcription when done
# flood_stream.stop_transcript()

---

## 📦 Step 5: Define a Flash Flood Event

Now, we’ll define an event type in the system to detect visual signs of a flash flood.


In [ ]:
flood_event_id = conn.create_event(
    event_prompt="Detect sudden flash floods or water surges.",
    label="flash_flood"
)
print("Event ID:", flood_event_id)


Event ID: 43a5c24f1e18df54


---
## 📦 Step 6: Attach an Alert for the Flash Flood Event

Alerts delivered instantly via WebSocket when a flood is detected.


In [ ]:
import os

RTSTREAM_ALERT_CALLBACK_URL = "https://example.com"
flood_alert_id = flood_index.create_alert(
    flood_event_id,
    callback_url=RTSTREAM_ALERT_CALLBACK_URL,
    ws_connection_id=ws.connection_id
)
print("Alert ID:", flood_alert_id)


Alert ID: 6a10bb3ff4fe4585


---
## 📦 Step 7: Listen for Flash Flood Alerts

Listen for incoming WebSocket alerts.


In [ ]:
import json

flood_alerts = []

async def listen_for_flood_alerts():
    timeout = 30
    print(f"Listening for flood alerts ({timeout} seconds)...")
    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if not isinstance(msg, dict):
                    print("WebSocket message:", repr(msg))
                    continue
                if msg.get("channel") == "alert":
                    flood_alerts.append(msg)
                    print(f"\nALERT #{len(flood_alerts)} RECEIVED!")
                    print(json.dumps(msg, indent=2, default=str))
    except asyncio.TimeoutError:
        print(f"\nListening complete! {len(flood_alerts)} alert(s) received")

await listen_for_flood_alerts()


Listening for flood alerts (30 seconds)...

ALERT #1 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026-07-27T11:03:13.194529+00:00",
  "rtstream_id": "rts-019fa33a-b352-7c60-a51e-007acc0f652d",
  "rtstream_name": "Arizona Flood Stream",
  "data": {
    "event_id": "alert-6a10bb3ff4fe4585",
    "label": "flash_flood",
    "triggered": true,
    "confidence": 0.99,
    "start": 1785150182.125,
    "end": 1785150189.0650222,
    "player_url": "https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785150182000000-1785150190000000.m3u8",
    "stream_url": "https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785150182000000-1785150190000000.m3u8",
    "explanation": "The scene explicitly shows rapidly moving muddy water and foam surging through a dry rocky riverbed, which matches an active flash flood and the alert context."
  }
}

ALERT #2 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026

In [ ]:
if flood_alerts:
    first = flood_alerts[0]
    data = (first.get("data") or first) if isinstance(first, dict) else None
    data = data if isinstance(data, dict) else {}
    clip_url = data.get("stream_url")
    label = data.get("label", "alert")

    if isinstance(clip_url, str) and clip_url.strip():
        print(f"  Confidence : {data.get('confidence', 'N/A')}")
        print(f"  Explanation: {data.get('explanation', 'N/A')}\n")
        display(display_stream(clip_url, f"Arizona Desert · {label}"))
    else:
        print("Flood alert received without a playable stream_url. Raw payload:")
        print(json.dumps(first, indent=2, default=str))
else:
    print("No flood alerts received yet.")


  Confidence : 0.99
  Explanation: The scene explicitly shows rapidly moving muddy water and foam surging through a dry rocky riverbed, which matches an active flash flood and the alert context.



#### Disable the first-stage alert, then stop its index and source understanding before the separate monitoring lesson.

In [ ]:
flood_index.disable_alert(flood_alert_id)
print("Flood alert disabled")

Flood alert disabled


In [ ]:
flood_index.stop()
flood_understanding.stop()
print("First-stage index and understanding stopped")

First-stage index and understanding stopped


In [ ]:
await ws_wrapper.close()
print("WebSocket closed")

---

The first monitoring stage is now stopped before we begin the separate riverbed lesson.

In [ ]:
# The first-stage alert, index, and understanding remain stopped.

#### Optional reference: resume the source understanding and index before re-enabling its alert. Keep this block commented while continuing.

In [ ]:
# flood_understanding.start()
# flood_index.start()
# flood_index.enable_alert(flood_alert_id)

---
### Let us set up some other alerts that are necessary
1. Heavy rainfall, detecting heavy rainfall early can help us predict a flash flood occurence
2. Detect the presence of a person stuck in the flash flood, for immediate rescue  

We can start a separate continuous understanding job and index for monitoring rainfall or a person stuck in the area.

In [ ]:
riverbed_monitoring_understanding = flood_stream.understand(
    segmentation={"type": "time", "window": "5s"},
    analyzers=[
        {
            "type": "vlm",
            "name": "riverbed_scene",
            "sampling": {"frame_count": 3},
            "config": {
                "prompt": "Monitor the dry riverbed and surrounding area. If any person is detected, output 'Person detected, rescue needed'. Else if rainfall is detected, output 'Rainfall detected'. Otherwise, output 'No significant events detected.'"
            },
        }
    ],
    store=True,
)
riverbed_scene_output = riverbed_monitoring_understanding.outputs.get("riverbed_scene")

print("Continuous understanding ID:", riverbed_monitoring_understanding.id)

Continuous understanding ID: und-626f6e45ea05d887


### Create the Riverbed Monitoring Index

Create a continuous semantic index from the riverbed output for rainfall and rescue alerts.


In [ ]:
riverbed_monitoring_index = flood_stream.index(
    source=riverbed_scene_output,
    name="Riverbed_Monitoring_Index",
    use_for=["semantic"],
)
riverbed_monitoring_index_id = riverbed_monitoring_index.id
print("Continuous index ID:", riverbed_monitoring_index_id)


Continuous index ID: idx-ea27215cf452f140


#### Checking the list of indexes

In [ ]:
list_rtstream_indexes(flood_stream)

RTStreamSceneIndex:
            Index ID       : 38fa58f953099771
            RTStream ID    : rts-019ec145-ff1c-7903-9b07-7f728a38bcff
            Name           : Riverbed_Monitoring_Index
            Status         : running
            Config         : {'frame_count': '1', 'time': '15'}
            Prompt         : Monitor the dry riverbed and surrounding area. In case you detect heavy rainfall mention 'heavy rainfall detected'. If you detect a person stuck in the area during rainfall or flash flood mention 'person detected, rescue needed'
        
--------------------------------------------------------------------------------
RTStreamSceneIndex:
            Index ID       : 644e99cf63d55a7b
            RTStream ID    : rts-019ec145-ff1c-7903-9b07-7f728a38bcff
            Name           : Flash_Flood_Detection_Index
            Status         : running
            Config         : {'frame_count': '3', 'time': '5'}
            Prompt         : Monitor the dry riverbed and surroundi

---
### Let's inspect indexed records

In [ ]:
import json
import time

RECORD_LOOKBACK_SECONDS = 15 * 60

range_end = int(time.time())
range_start = range_end - RECORD_LOOKBACK_SECONDS
riverbed_records_payload = riverbed_monitoring_index.get_records(
    start=range_start,
    end=range_end,
    page=1,
    page_size=5,
)
print(json.dumps(riverbed_records_payload, indent=2, default=str))

{
  "next_page": true,
  "records": [
    {
      "description": "Rainfall detected",
      "end": 1785152106.1312373,
      "start": 1785152100.1012177
    },
    {
      "description": "Person detected, rescue needed",
      "end": 1785152114.1212633,
      "start": 1785152108.1012437
    },
    {
      "description": "Person detected, rescue needed",
      "end": 1785152122.1212895,
      "start": 1785152116.0912697
    },
    {
      "description": "Person detected, rescue needed",
      "end": 1785152130.1113155,
      "start": 1785152124.0812957
    },
    {
      "description": "No significant events detected.",
      "end": 1785152138.1113415,
      "start": 1785152132.0813217
    }
  ]
}


---
### Now we can setup events and alerts for the index

Re-establish WebSocket connection to ensure it's active

In [ ]:
ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()
print(f"WebSocket re-connected!")
print(f"Connection ID: {ws.connection_id}")

INFO:videodb.websocket_client:WebSocket connected with ID: gVXu9I5kleO4KEhCkA==


WebSocket re-connected!
Connection ID: gVXu9I5kleO4KEhCkA==


1. Rainfall Detection

In [ ]:
# Create rainfall event
rainfall_event_id = conn.create_event(
    event_prompt="Detect heavy rainfall.",
    label="heavy_rainfall"
)
print("Event ID:", rainfall_event_id)

Event ID: 1a58dd6113a8a0d0


In [ ]:
rainfall_alert_id = riverbed_monitoring_index.create_alert(
    rainfall_event_id,
    callback_url=RTSTREAM_ALERT_CALLBACK_URL,
    ws_connection_id=ws.connection_id
)
print("Rainfall Alert ID:", rainfall_alert_id)


Rainfall Alert ID: 500f57159b90d976


---
2. Human detection for rescue

In [ ]:
# Create rescue event
rescue_event_id = conn.create_event(
    event_prompt="Detect if there is a person",
    label="human_rescue"
)
print("Event ID:", rescue_event_id)

Event ID: a4b0e2b901c4346a


In [ ]:
rescue_alert_id = riverbed_monitoring_index.create_alert(
    rescue_event_id,
    callback_url=RTSTREAM_ALERT_CALLBACK_URL,
    ws_connection_id=ws.connection_id
)
print("Rescue Alert ID:", rescue_alert_id)


Rescue Alert ID: 657d632d50778585


---
### Let us see the list of alerts associated with the `riverbed_monitoring_index`

In [ ]:
def list_rtstream_alerts(rtstream, index_id):
    rtstream_index = rtstream.get_index(index_id)
    alerts = rtstream_index.list_alerts()
    print(json.dumps(alerts, indent=2, default=str))

list_rtstream_alerts(flood_stream, riverbed_monitoring_index_id)


[
  {
    "alert_id": "657d632d50778585",
    "callback_url": "https://example.com",
    "event_id": "a4b0e2b901c4346a",
    "label": "human_rescue",
    "prompt": "Detect if there is a person",
    "status": "enabled",
    "ws_connection_id": "gVXu9I5kleO4KEhCkA=="
  },
  {
    "alert_id": "b887dcafc012c957",
    "callback_url": "https://example.com",
    "event_id": "fc946946e7bfad2f",
    "label": "human_rescue",
    "prompt": "Detect if there is a person",
    "status": "disabled",
    "ws_connection_id": "gVXo8E6h9eO4KEigLA=="
  }
]


---
## 📦 Step 8: Listen for Riverbed Monitoring Alerts

Listen for `heavy_rainfall` and `human_rescue` alerts via WebSocket.


In [ ]:
import json

riverbed_alerts = []

async def listen_for_riverbed_alerts():
    timeout = 120
    print(f"Listening for riverbed alerts ({timeout} seconds)...")
    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if not isinstance(msg, dict):
                    print("WebSocket message:", repr(msg))
                    continue
                if msg.get("channel") == "alert":
                    riverbed_alerts.append(msg)
                    print(f"\nALERT #{len(riverbed_alerts)} RECEIVED!")
                    print(json.dumps(msg, indent=2, default=str))
    except asyncio.TimeoutError:
        print(f"\nListening complete! {len(riverbed_alerts)} alert(s) received")
    except Exception as e:
        print(f"\nAn error occurred while listening for alerts: {e}")

await listen_for_riverbed_alerts()

Listening for riverbed alerts (120 seconds)...

ALERT #1 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026-07-27T11:46:25.996761+00:00",
  "rtstream_id": "rts-019fa33a-b352-7c60-a51e-007acc0f652d",
  "rtstream_name": "Arizona Flood Stream",
  "data": {
    "event_id": "alert-500f57159b90d976",
    "label": "heavy_rainfall",
    "triggered": true,
    "confidence": 0.97,
    "start": 1785152775.113424,
    "end": 1785152782.0534468,
    "player_url": "https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785152775000000-1785152783000000.m3u8",
    "stream_url": "https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785152775000000-1785152783000000.m3u8",
    "explanation": "A person is detected in a heavy rainfall scene, which suggests potential immediate safety risk and need for rescue assistance."
  }
}

ALERT #2 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026-07-27T11:46:27.748512+

In [ ]:
if riverbed_alerts:
    first = riverbed_alerts[0]
    data = (first.get("data") or first) if isinstance(first, dict) else None
    data = data if isinstance(data, dict) else {}
    clip_url = data.get("stream_url")
    label = data.get("label", "alert")

    if isinstance(clip_url, str) and clip_url.strip():
        display(display_stream(clip_url, f"Arizona Desert · {label}"))
    else:
        print("Riverbed alert received without a playable stream_url. Raw payload:")
        print(json.dumps(first, indent=2, default=str))
else:
    print("No riverbed alerts received yet.")


---

## 📡 Example Alert Payload

This illustrative payload shows fields a live alert may provide. Inspect the raw WebSocket message before using optional fields:

```json
{
  "channel": "alert",
  "timestamp": "2026-07-27T11:48:19.327265+00:00",
  "rtstream_id": "rts-019fa33a-b352-7c60-a51e-007acc0f652d",
  "rtstream_name": "Arizona Flood Stream",
  "data": {
    "event_id": "alert-657d632d50778585",
    "label": "human_rescue",
    "triggered": true,
    "confidence": 0.98,
    "start": 1785152886.1337905,
    "end": 1785152893.0738134,
    "player_url": "https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785152886000000-1785152894000000.m3u8",
    "stream_url": "https://rt.stream.videodb.io/manifests/rts-019fa33a-b352-7c60-a51e-007acc0f652d/1785152886000000-1785152894000000.m3u8",
    "explanation": "A person is detected in the scene, and the alert context indicates that detection of a person should trigger a rescue-related alert."
  }
}
```

**💡 What just happened?**

- We connected to VideoDB's WebSocket for real-time event delivery
- Created alerts for flash floods, heavy rainfall, and human rescue detection
- Listened for 30 seconds and automatically received alerts
- When an alert includes a playable `stream_url`, the matching evidence can be replayed

**⚡ Why WebSockets?**
- **Instant delivery**: Alerts arrive in real-time as events happen
- **Live dashboards**: Perfect for building interactive monitoring interfaces
- **Two-way communication**: Can send and receive data on the same connection

> **💡 Callback + WebSocket delivery**  
> Configure a public callback URL for alert delivery, and also pass `ws_connection_id` to receive the same alerts over the WebSocket.

---
-  Let us disable the alerts now.

In [ ]:
riverbed_monitoring_index.disable_alert(rainfall_alert_id)
riverbed_monitoring_index.disable_alert(rescue_alert_id)
print("Riverbed alerts disabled")


Riverbed alerts disabled


- Stop the remaining continuous index, its source understanding, and the stream before closing the WebSocket.

In [ ]:
riverbed_monitoring_index.stop()
riverbed_monitoring_understanding.stop()
print("Remaining index and understanding stopped")

flood_stream.stop()
print("Stream stopped")

await ws_wrapper.close()
print("WebSocket closed")


Remaining index and understanding stopped
Stream stopped
WebSocket closed


<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> You've built a complete AI-powered flash flood detection system — from live stream to real-time WebSocket alerts!
</div>

---

## 🌙 Wrapping Up: Real-Time Environmental Safety

With this system in place, communities, tourists, and local authorities in Arizona’s desert regions can receive **immediate alerts** when a dangerous flash flood occurs — gaining critical seconds to take cover, clear routes, or initiate rescues.

---

## 🔥 What Else Could We Monitor?

This system isn’t limited to flash floods. The same AI-driven video monitoring approach can protect lives in other natural disasters too.

---

### 🌲 Forest Fire Detection

**Why it’s needed:**  
In remote forest areas, smoke plumes and early fire flickers often go unnoticed for several minutes before traditional sensors or satellites pick them up.

**How we can do it better:**  
With AI-powered cameras watching key zones, we can detect rapid smoke build-ups or visible flames long before automated sensors trigger.

**Example indexing prompt:**

```markdown
"Monitor the forest area carefully. Detect sudden rising smoke, increasing haze, or visible flames. Clearly describe if a fire outbreak is visibly starting."
```

---

### 🌊 Sudden Tsunami Detection

**Why it’s needed:**  
Even with tsunami sensors and ocean buoys, near-shoreline towns often get only minutes of warning. Visual cues like water rapidly pulling away from shore or unusually large approaching waves are immediate, reliable signs.

**How we can do it better:**  
Install AI cameras on popular beaches and coastlines to detect rapid water retreat or walls of water approaching.

**Example indexing prompt:**

```markdown
"Watch the beach and shoreline closely. Detect unusual rapid water withdrawal from the shore, or approaching large waves indicating a possible tsunami. Describe the situation clearly."
```

---

### 🏔️ Landslide / Avalanche Detection

**Why it’s needed:**  
Mountain highways and tourist trails are often cut off by sudden landslides or snow avalanches, where every second matters for evacuation and road closures.

**How we can do it better:**  
Place AI cameras at landslide-prone mountain slopes or snow-covered passes to detect falling rocks, moving soil, or snow slides in real time.

**Example indexing prompt:**

```markdown
"Monitor the mountain slope area carefully. Detect falling rocks, soil displacement, or sudden snow slides. Clearly describe if a landslide or avalanche is starting."
```

---

## 🌍 The Possibilities Are Endless

Real-time AI video monitoring isn’t just for cities and homes — it can actively save lives in wild, unpredictable environments too.

**What natural threat would *you* monitor next?**